# Edge Deployment: Running Models on Constrained Devices

Edge deployment means running machine learning models directly on the device where data is generated, rather than sending that data to a remote server. This includes mobile phones, IoT sensors, embedded systems, smart cameras, and web browsers.

In this notebook you will learn:
- Why edge deployment matters and when to use it
- How to export models to ONNX format for universal compatibility
- How to run inference with ONNX Runtime
- How to apply INT8 quantization to shrink models and speed up inference
- How to benchmark latency and memory usage

## Why Edge Deployment?

When you run a model on a remote server the data must travel over a network, the server must process it, and the result must travel back. For many applications this is perfectly fine. But in other cases it creates real problems.

**Latency**: A network round trip adds 20-200ms on a good connection, much more on cellular or spotty Wi-Fi. Real-time applications like gesture recognition, driver assistance, or industrial fault detection cannot tolerate that delay. Running the model on the device eliminates the round trip entirely.

**Privacy**: Sending data to a server means the data leaves the device. For medical images, voice recordings, or financial data, privacy regulations or user expectations may require that raw data never leaves the device. On-device inference keeps sensitive data local.

**Offline capability**: Mobile apps, field equipment, and remote sensors often operate without a reliable internet connection. A model running on the device works whether or not the network is available.

**Cost at scale**: Millions of inference calls per day to a cloud server can cost tens of thousands of dollars per month. On-device inference moves that cost to the hardware purchase price, which may already be paid.

The tradeoff is that edge devices have strict constraints you must design around.

## Edge Device Constraints

| Constraint | Typical Range | Implication |
|---|---|---|
| RAM | 256 MB - 4 GB | Model weights + activations must fit |
| Compute | No GPU or small GPU | Inference must be fast on CPU |
| Power / Battery | 1-5W budget | Fewer floating point ops = less power |
| Storage | 64 MB - 2 GB for model | Smaller model = more space for app |
| OS | Android, iOS, Linux embedded | Framework must support target OS |

A GPT-scale model with billions of parameters is impossible on most edge devices. But a ResNet-50 (25M params, ~98MB in FP32) can run on a mid-range phone after quantization to ~25MB. Smaller models like MobileNet or EfficientNet-Lite were designed specifically for edge constraints.

In [1]:
import numpy as np
import time
import os
import tracemalloc

print('NumPy version:', np.__version__)

import onnxruntime as ort
print('ONNX Runtime version:', ort.__version__)
print('Available providers:', ort.get_available_providers())

import sklearn
print('scikit-learn version:', sklearn.__version__)

import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

NumPy version: 2.5.0
ONNX Runtime version: 1.27.0
Available providers: ['AzureExecutionProvider', 'CPUExecutionProvider']


scikit-learn version: 1.9.0


PyTorch version: 2.12.1+cpu
CUDA available: False


## Section 1: Model Optimization for Edge

Three main techniques reduce model size and inference cost for edge deployment.

### Quantization

Standard deep learning uses 32-bit floating point (FP32) for weights and activations. Quantization converts these to lower precision:

- **FP16**: half precision, 2x size reduction, nearly identical accuracy, supported by many GPUs
- **INT8**: 8-bit integer, 4x size reduction, 2-4x speed increase on CPU, typically less than 1% accuracy drop
- **INT4**: 4x further reduction, more accuracy loss, used in extreme constrained scenarios

INT8 quantization is the first optimization to try because the size and speed gains are large and the accuracy loss is usually negligible for most tasks.

### Pruning

Many weights in a trained model are very close to zero. Pruning sets these to exactly zero (structured or unstructured sparsity). Structured pruning removes entire channels or layers, which leads to actual speedups. Unstructured pruning creates sparse weight matrices that require specialized hardware or libraries to accelerate.

### Knowledge Distillation

A large, accurate "teacher" model trains a smaller "student" model to mimic its output distributions (soft labels), not just the hard class labels. The student learns to match the teacher's confidence patterns across all classes, which provides richer training signal than one-hot labels. This often produces a small model that outperforms a model of the same size trained from scratch.

In [2]:
# Demonstrate the size impact of precision
import sys

n_params = 1_000_000  # 1M parameter model

size_fp32 = n_params * 4  # 4 bytes per float32
size_fp16 = n_params * 2  # 2 bytes per float16
size_int8 = n_params * 1  # 1 byte per int8
size_int4 = n_params * 0.5  # 0.5 bytes per int4

def to_mb(b):
    return b / (1024 * 1024)

print('For a 1M parameter model:')
print(f'  FP32: {to_mb(size_fp32):.2f} MB (baseline)')
print(f'  FP16: {to_mb(size_fp16):.2f} MB ({size_fp32 / size_fp16:.0f}x smaller)')
print(f'  INT8: {to_mb(size_int8):.2f} MB ({size_fp32 / size_int8:.0f}x smaller)')
print(f'  INT4: {to_mb(size_int4):.2f} MB ({size_fp32 / size_int4:.0f}x smaller)')

print('\nFor a 100M parameter model (e.g., ResNet-50 equivalent):')
n = 100_000_000
print(f'  FP32: {to_mb(n*4):.1f} MB')
print(f'  INT8: {to_mb(n*1):.1f} MB')

For a 1M parameter model:
  FP32: 3.81 MB (baseline)
  FP16: 1.91 MB (2x smaller)
  INT8: 0.95 MB (4x smaller)
  INT4: 0.48 MB (8x smaller)

For a 100M parameter model (e.g., ResNet-50 equivalent):
  FP32: 381.5 MB
  INT8: 95.4 MB


## Section 2: ONNX Export

ONNX (Open Neural Network Exchange) is an open format for representing machine learning models. It acts as a universal intermediate representation:

- **Train** in PyTorch, TensorFlow, scikit-learn, XGBoost, or any framework
- **Export** to ONNX once
- **Run** on any platform: Linux, Windows, macOS, Android, iOS, WebAssembly

ONNX Runtime is a high-performance inference engine that reads ONNX models and runs them efficiently on whatever hardware is available.

In [3]:
# Export a PyTorch model to ONNX
import torch
import torch.nn as nn

# Define a simple feed-forward model representative of a real task
class TabularClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, n_classes)
        )
    
    def forward(self, x):
        return self.net(x)

INPUT_DIM = 20
HIDDEN_DIM = 128
N_CLASSES = 3

model = TabularClassifier(INPUT_DIM, HIDDEN_DIM, N_CLASSES)
model.eval()  # Switch to inference mode: disables dropout, freezes BatchNorm stats

print('Model architecture:')
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters: {n_params:,}')

Model architecture:
TabularClassifier(
  (net): Sequential(
    (0): Linear(in_features=20, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=3, bias=True)
  )
)

Total parameters: 11,139


In [4]:
import tempfile

# Create a temporary directory to store our model files
tmpdir = tempfile.mkdtemp()
onnx_path = os.path.join(tmpdir, 'tabular_model.onnx')
onnx_quant_path = os.path.join(tmpdir, 'tabular_model_int8.onnx')

# Export to ONNX
# torch.onnx.export traces the model by running it with a dummy input
# It records all operations and their shapes into the ONNX graph
dummy_input = torch.randn(1, INPUT_DIM)  # batch_size=1, input_dim=20

torch.onnx.export(
    model,                      # PyTorch model
    dummy_input,                # example input (defines input shape)
    onnx_path,                  # output file path
    input_names=['input'],      # name the input tensor
    output_names=['output'],    # name the output tensor
    dynamic_axes={              # allow variable batch size at runtime
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    },
    opset_version=17            # ONNX opset version (17 = current stable)
)

model_size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
print(f'ONNX model exported to: {onnx_path}')
print(f'Model size: {model_size_mb:.3f} MB')

print('\n--- Exporting sklearn model to ONNX (pattern) ---')
print("""
# Pattern for sklearn export (requires skl2onnx):
# from skl2onnx import to_onnx
# from skl2onnx.common.data_types import FloatTensorType
#
# clf = RandomForestClassifier().fit(X_train, y_train)
# initial_type = [('float_input', FloatTensorType([None, X_train.shape[1]]))]
# onnx_model = to_onnx(clf, initial_types=initial_type)
# with open('sklearn_model.onnx', 'wb') as f:
#     f.write(onnx_model.SerializeToString())
""")

/tmp/ipykernel_34645/2699891197.py:13: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0629 12:57:23.534000 34645 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `TabularClassifier([...]` with `torch.export.export(..., strict=False)`...


[torch.onnx] Obtain model graph for `TabularClassifier([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
ONNX model exported to: /tmp/tmpuz0dj1cs/tabular_model.onnx
Model size: 0.002 MB

--- Exporting sklearn model to ONNX (pattern) ---

# Pattern for sklearn export (requires skl2onnx):
# from skl2onnx import to_onnx
# from skl2onnx.common.data_types import FloatTensorType
#
# clf = RandomForestClassifier().fit(X_train, y_train)
# initial_type = [('float_input', FloatTensorType([None, X_train.shape[1]]))]
# onnx_model = to_onnx(clf, initial_types=initial_type)
# with open('sklearn_model.onnx', 'wb') as f:
#     f.write(onnx_model.SerializeToString())



## Section 3: ONNX Runtime Inference

ONNX Runtime (ORT) loads an ONNX file and runs inference. It selects the best execution provider available on the current machine:

- `CPUExecutionProvider`: always available, uses CPU with SIMD optimizations
- `CUDAExecutionProvider`: NVIDIA GPU acceleration
- `CoreMLExecutionProvider`: Apple Neural Engine on macOS/iOS
- `TensorrtExecutionProvider`: NVIDIA TensorRT for maximum GPU throughput

ORT applies graph optimizations automatically (operator fusion, constant folding) that speed up inference beyond what PyTorch would do in eval mode.

In [5]:
import onnxruntime as ort

# Create an inference session
# ORT loads and optimizes the graph at session creation time
sess = ort.InferenceSession(
    onnx_path,
    providers=['CPUExecutionProvider']  # explicit provider list
)

# Inspect model inputs and outputs
print('Model inputs:')
for inp in sess.get_inputs():
    print(f'  name={inp.name}, shape={inp.shape}, dtype={inp.type}')

print('\nModel outputs:')
for out in sess.get_outputs():
    print(f'  name={out.name}, shape={out.shape}, dtype={out.type}')

# Run inference
# ORT expects numpy arrays, not PyTorch tensors
sample_input = np.random.randn(4, INPUT_DIM).astype(np.float32)  # batch of 4

outputs = sess.run(
    None,                          # None = return all outputs
    {'input': sample_input}        # dict mapping input name to array
)

logits = outputs[0]
print(f'\nInput shape: {sample_input.shape}')
print(f'Output shape: {logits.shape}')
print(f'Output (logits):\n{logits}')
print(f'Predicted classes: {np.argmax(logits, axis=1)}')

Model inputs:
  name=input, shape=['batch_size', 20], dtype=tensor(float)

Model outputs:
  name=output, shape=['batch_size', 3], dtype=tensor(float)

Input shape: (4, 20)
Output shape: (4, 3)
Output (logits):
[[-0.2353696   0.18165652 -0.15152906]
 [-0.22678763 -0.00195926 -0.13964206]
 [-0.139992    0.20974207 -0.05350919]
 [-0.17140913  0.04439946 -0.01812657]]
Predicted classes: [1 1 1 1]


In [6]:
# Speed comparison: PyTorch vs ONNX Runtime
N_RUNS = 500
BATCH = 32
test_input_np = np.random.randn(BATCH, INPUT_DIM).astype(np.float32)
test_input_torch = torch.from_numpy(test_input_np)

# Warm up both runtimes (first call initializes caches)
with torch.no_grad():
    _ = model(test_input_torch)
_ = sess.run(None, {'input': test_input_np})

# Benchmark PyTorch (CPU)
with torch.no_grad():
    t0 = time.perf_counter()
    for _ in range(N_RUNS):
        _ = model(test_input_torch)
    pytorch_ms = (time.perf_counter() - t0) * 1000 / N_RUNS

# Benchmark ONNX Runtime (CPU)
t0 = time.perf_counter()
for _ in range(N_RUNS):
    _ = sess.run(None, {'input': test_input_np})
ort_ms = (time.perf_counter() - t0) * 1000 / N_RUNS

print(f'Batch size: {BATCH}, Runs: {N_RUNS}')
print(f'PyTorch CPU:       {pytorch_ms:.3f} ms/batch')
print(f'ONNX Runtime CPU:  {ort_ms:.3f} ms/batch')
speedup = pytorch_ms / ort_ms
print(f'ORT speedup:       {speedup:.2f}x faster')

Batch size: 32, Runs: 500
PyTorch CPU:       0.290 ms/batch
ONNX Runtime CPU:  0.058 ms/batch
ORT speedup:       4.99x faster


## Section 4: Quantization with ONNX Runtime

ONNX Runtime's quantization tools apply INT8 quantization directly to an ONNX model without retraining. There are two modes:

- **Dynamic quantization**: weights are quantized offline; activations are quantized at runtime per-batch. Fast to apply, no calibration data needed. Good for NLP models and tabular models.
- **Static quantization**: both weights and activations are quantized offline using a small calibration dataset. Requires representative data. Better for CNNs.

Dynamic quantization is the easiest starting point and often sufficient.

In [7]:
from onnxruntime.quantization import quantize_dynamic, QuantType
from onnxruntime.quantization import shape_inference as quant_shape_inference

# Pre-process: run shape inference so quantize_dynamic can resolve all shapes
pre_processed_path = os.path.join(tmpdir, 'tabular_model_pre.onnx')
quant_shape_inference.quant_pre_process(onnx_path, pre_processed_path, skip_optimization=False)

# Apply INT8 dynamic quantization
# This replaces Linear (MatMul + Add) operations with INT8 versions
quantize_dynamic(
    model_input=pre_processed_path,
    model_output=onnx_quant_path,
    weight_type=QuantType.QInt8   # INT8 weights
)

# Compare file sizes
orig_size = os.path.getsize(onnx_path)
quant_size = os.path.getsize(onnx_quant_path)

print('File size comparison:')
print(f'  FP32 ONNX:  {orig_size / 1024:.1f} KB')
print(f'  INT8 ONNX:  {quant_size / 1024:.1f} KB')
print(f'  Reduction:  {(1 - quant_size/orig_size)*100:.1f}%')

print(f'\nOriginal path: {onnx_path}')
print(f'Quantized path: {onnx_quant_path}')

File size comparison:
  FP32 ONNX:  1.6 KB
  INT8 ONNX:  14.7 KB
  Reduction:  -809.9%

Original path: /tmp/tmpuz0dj1cs/tabular_model.onnx
Quantized path: /tmp/tmpuz0dj1cs/tabular_model_int8.onnx


In [8]:
# Benchmark original vs quantized
sess_quant = ort.InferenceSession(
    onnx_quant_path,
    providers=['CPUExecutionProvider']
)

# Warm up
_ = sess.run(None, {'input': test_input_np})
_ = sess_quant.run(None, {'input': test_input_np})

# Benchmark FP32
t0 = time.perf_counter()
for _ in range(N_RUNS):
    out_fp32 = sess.run(None, {'input': test_input_np})
ms_fp32 = (time.perf_counter() - t0) * 1000 / N_RUNS

# Benchmark INT8
t0 = time.perf_counter()
for _ in range(N_RUNS):
    out_int8 = sess_quant.run(None, {'input': test_input_np})
ms_int8 = (time.perf_counter() - t0) * 1000 / N_RUNS

# Accuracy comparison: compare logits between FP32 and INT8
logits_fp32 = out_fp32[0]
logits_int8 = out_int8[0]
pred_fp32 = np.argmax(logits_fp32, axis=1)
pred_int8 = np.argmax(logits_int8, axis=1)
agreement = np.mean(pred_fp32 == pred_int8)
max_diff = np.max(np.abs(logits_fp32 - logits_int8))

print('Performance comparison:')
print(f'  FP32 inference: {ms_fp32:.3f} ms/batch')
print(f'  INT8 inference: {ms_int8:.3f} ms/batch')
print(f'  Speedup:        {ms_fp32/ms_int8:.2f}x')
print(f'\nAccuracy:')
print(f'  Prediction agreement: {agreement*100:.1f}%')
print(f'  Max logit difference: {max_diff:.4f}')

Performance comparison:
  FP32 inference: 0.031 ms/batch
  INT8 inference: 0.041 ms/batch
  Speedup:        0.75x

Accuracy:
  Prediction agreement: 100.0%
  Max logit difference: 0.0386


## Section 5: TFLite and CoreML (Concepts)

Beyond ONNX Runtime, two platform-specific formats dominate edge deployment.

### TensorFlow Lite (TFLite)

TFLite is Google's format for Android, embedded Linux, and microcontrollers. The model is stored in the FlatBuffer format (.tflite file), which can be memory-mapped directly without parsing overhead.

```python
# Pattern: Convert ONNX to TFLite via tf2onnx / onnx-tf
# Step 1: ONNX -> TensorFlow SavedModel
# onnx-tf convert -i model.onnx -o saved_model/
# Step 2: TF SavedModel -> TFLite
# converter = tf.lite.TFLiteConverter.from_saved_model('saved_model/')
# converter.optimizations = [tf.lite.Optimize.DEFAULT]  # INT8
# tflite_model = converter.convert()
# with open('model.tflite', 'wb') as f:
#     f.write(tflite_model)

# Runtime inference:
# interp = tf.lite.Interpreter(model_path='model.tflite')
# interp.allocate_tensors()
# input_details = interp.get_input_details()
# interp.set_tensor(input_details[0]['index'], input_data)
# interp.invoke()
# output = interp.get_tensor(output_details[0]['index'])
```

TFLite supports a delegate system for hardware acceleration: GPU delegate, NNAPI delegate (Android Neural Networks API), Hexagon DSP delegate (Qualcomm chips), and Edge TPU delegate (Google Coral).

### CoreML

CoreML is Apple's format for iOS, macOS, watchOS, and tvOS. Models use the .mlmodel (or .mlpackage) format and run on the Apple Neural Engine, GPU, or CPU automatically.

```python
# Pattern: Convert ONNX to CoreML
# import coremltools as ct
# import onnx

# onnx_model = onnx.load('model.onnx')
# coreml_model = ct.convert(
#     onnx_model,
#     inputs=[ct.TensorType(shape=(1, 20))]
# )
# coreml_model.save('model.mlmodel')

# On-device (Swift):
# let model = try MLModel(contentsOf: modelURL)
# let prediction = try model.prediction(from: input)
```

CoreML automatically routes computation to the most efficient processor available on the device, providing significant power efficiency over CPU-only inference.

In [9]:
# Conceptual summary: ecosystem mapping
ecosystem = {
    'Android / embedded Linux': {
        'format': 'TFLite (.tflite)',
        'runtime': 'TFLite Interpreter',
        'accelerators': 'NNAPI, GPU delegate, Hexagon DSP'
    },
    'Apple devices (iOS/macOS)': {
        'format': 'CoreML (.mlmodel)',
        'runtime': 'CoreML Framework',
        'accelerators': 'Apple Neural Engine, Metal GPU'
    },
    'Cross-platform (any OS)': {
        'format': 'ONNX (.onnx)',
        'runtime': 'ONNX Runtime',
        'accelerators': 'CUDA, DirectML, CoreML, NNAPI providers'
    },
    'Browser (WebAssembly)': {
        'format': 'ONNX or TFLite',
        'runtime': 'ONNX Runtime Web / TensorFlow.js',
        'accelerators': 'WebGL, WebGPU'
    },
    'Microcontrollers (<1MB RAM)': {
        'format': 'TFLite Micro (.tflite)',
        'runtime': 'TFLite Micro runtime (C++)',
        'accelerators': 'Custom kernels per MCU'
    }
}

for platform, info in ecosystem.items():
    print(f'\n{platform}:')
    for k, v in info.items():
        print(f'  {k}: {v}')


Android / embedded Linux:
  format: TFLite (.tflite)
  runtime: TFLite Interpreter
  accelerators: NNAPI, GPU delegate, Hexagon DSP

Apple devices (iOS/macOS):
  format: CoreML (.mlmodel)
  runtime: CoreML Framework
  accelerators: Apple Neural Engine, Metal GPU

Cross-platform (any OS):
  format: ONNX (.onnx)
  runtime: ONNX Runtime
  accelerators: CUDA, DirectML, CoreML, NNAPI providers

Browser (WebAssembly):
  format: ONNX or TFLite
  runtime: ONNX Runtime Web / TensorFlow.js
  accelerators: WebGL, WebGPU

Microcontrollers (<1MB RAM):
  format: TFLite Micro (.tflite)
  runtime: TFLite Micro runtime (C++)
  accelerators: Custom kernels per MCU


## Section 6: Benchmarking on Edge

Benchmarking on your development machine is not representative of the target device. That said, understanding how to measure latency and memory usage correctly is essential before you deploy.

### Latency Benchmarking

Always:
1. **Warm up** the runtime (run a few inferences before measuring)
2. **Average over many runs** (100+ for sub-millisecond operations)
3. **Report percentiles** (p50, p95, p99) not just mean, because tail latency matters for user experience

### Memory Profiling

Use `tracemalloc` (Python standard library) for heap allocation tracking. On real devices you would use platform-specific tools (Android Studio profiler, Instruments on macOS).

In [10]:
def benchmark_inference(session, input_data, n_runs=200, input_name='input'):
    """Benchmark inference latency with warmup and percentile reporting."""
    # Warmup
    for _ in range(10):
        session.run(None, {input_name: input_data})
    
    # Measure
    latencies = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        session.run(None, {input_name: input_data})
        latencies.append((time.perf_counter() - t0) * 1000)
    
    latencies = np.array(latencies)
    return {
        'mean_ms': float(np.mean(latencies)),
        'p50_ms': float(np.percentile(latencies, 50)),
        'p95_ms': float(np.percentile(latencies, 95)),
        'p99_ms': float(np.percentile(latencies, 99)),
        'min_ms': float(np.min(latencies)),
        'max_ms': float(np.max(latencies))
    }

single_input = np.random.randn(1, INPUT_DIM).astype(np.float32)  # batch=1 for latency

stats_fp32 = benchmark_inference(sess, single_input)
stats_int8 = benchmark_inference(sess_quant, single_input)

print('Latency benchmark (batch=1, single request):')
print(f'\n{"Metric":<15} {"FP32 (ms)":>12} {"INT8 (ms)":>12} {"Speedup":>10}')
print('-' * 52)
for key in ['mean_ms', 'p50_ms', 'p95_ms', 'p99_ms']:
    label = key.replace('_ms', '')
    fp32_val = stats_fp32[key]
    int8_val = stats_int8[key]
    speedup = fp32_val / int8_val if int8_val > 0 else float('nan')
    print(f'{label:<15} {fp32_val:>12.4f} {int8_val:>12.4f} {speedup:>10.2f}x')

Latency benchmark (batch=1, single request):

Metric             FP32 (ms)    INT8 (ms)    Speedup
----------------------------------------------------
mean                  0.0202       0.0325       0.62x
p50                   0.0152       0.0292       0.52x
p95                   0.0288       0.0418       0.69x
p99                   0.0402       0.2850       0.14x


In [11]:
import tracemalloc

def measure_memory(session, input_data, input_name='input'):
    """Measure peak memory allocation during inference."""
    tracemalloc.start()
    for _ in range(50):
        session.run(None, {input_name: input_data})
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak / (1024 * 1024)  # MB

mem_fp32 = measure_memory(sess, single_input)
mem_int8 = measure_memory(sess_quant, single_input)

print('Memory usage during inference (Python heap allocations):')
print(f'  FP32 model: {mem_fp32:.2f} MB peak')
print(f'  INT8 model: {mem_int8:.2f} MB peak')

print('\nModel file sizes:')
print(f'  FP32: {os.path.getsize(onnx_path) / 1024:.1f} KB')
print(f'  INT8: {os.path.getsize(onnx_quant_path) / 1024:.1f} KB')

print('\nNote: For real device profiling use:')
print('  Android: Android Studio Memory Profiler')
print('  iOS/macOS: Instruments -> Leaks / Allocations')
print('  Linux embedded: valgrind --tool=massif, /proc/self/status')

Memory usage during inference (Python heap allocations):
  FP32 model: 0.01 MB peak
  INT8 model: 0.00 MB peak

Model file sizes:
  FP32: 1.6 KB
  INT8: 14.7 KB

Note: For real device profiling use:
  Android: Android Studio Memory Profiler
  iOS/macOS: Instruments -> Leaks / Allocations
  Linux embedded: valgrind --tool=massif, /proc/self/status


## Section 7: Real Use Cases

### Offline Speech Recognition

Voice assistants that work without internet (airplane mode, areas with no coverage). Models like Whisper can be quantized to run on-device. The user's voice recording never leaves the phone.

### Real-Time Image Classification on Camera

Classifying objects frame-by-frame at 30 fps requires less than ~33ms per frame. On a modern phone with a neural engine, MobileNetV3 achieves this comfortably. Applications include accessibility tools (real-time scene description), augmented reality overlays, and barcode reading.

### Predictive Maintenance on IoT

Industrial equipment (motors, pumps, CNC machines) has vibration sensors. A small anomaly detection model running on an edge gateway or directly on the sensor can detect bearing faults before they cause failures. Sending raw sensor streams (10kHz) to the cloud is expensive; sending just an anomaly alert is cheap.

### Medical Devices

Blood glucose monitors, ECG patches, and pulse oximeters all run ML on small microcontrollers. Regulatory requirements (FDA, CE) often require that the inference happens on a certified device without cloud dependencies.

### Autonomous Vehicles

Decision latency in a self-driving car must be sub-10ms. Sending camera frames to the cloud and back is impossible at highway speeds. All perception (object detection, lane keeping, depth estimation) runs on purpose-built edge chips (NVIDIA Drive, Tesla FSD chip, Mobileye EyeQ).

In [12]:
# Simulate an edge inference loop: batch=1, continuous stream
print('Simulating edge inference loop (streaming sensor data)...')

N_SAMPLES = 100
results = []
latencies = []

for i in range(N_SAMPLES):
    # Simulate arriving sensor data
    sensor_reading = np.random.randn(1, INPUT_DIM).astype(np.float32)
    
    t0 = time.perf_counter()
    output = sess_quant.run(None, {'input': sensor_reading})
    elapsed_ms = (time.perf_counter() - t0) * 1000
    
    predicted_class = int(np.argmax(output[0]))
    confidence = float(np.max(output[0]))  # raw logit as proxy
    
    results.append(predicted_class)
    latencies.append(elapsed_ms)

latencies_arr = np.array(latencies)
print(f'\nProcessed {N_SAMPLES} samples with INT8 model:')
print(f'  Mean latency:  {np.mean(latencies_arr):.3f} ms')
print(f'  P99 latency:   {np.percentile(latencies_arr, 99):.3f} ms')
print(f'  Throughput:    {1000 / np.mean(latencies_arr):.0f} samples/sec')
print(f'\nClass distribution: {np.bincount(np.array(results))}')

Simulating edge inference loop (streaming sensor data)...

Processed 100 samples with INT8 model:
  Mean latency:  0.021 ms
  P99 latency:   0.032 ms
  Throughput:    47194 samples/sec

Class distribution: [  0 100]


## Key Takeaways

**ONNX is the universal intermediate format.** Train in any framework, export once, run anywhere. This separates training infrastructure from deployment infrastructure.

**Quantize to INT8 first.** Dynamic INT8 quantization with `onnxruntime.quantization.quantize_dynamic()` is a single function call that typically yields 4x size reduction and 1.5-3x speedup with less than 1% accuracy loss.

**ONNX Runtime is the most portable inference engine.** It supports CPUExecutionProvider on every platform and optional hardware acceleration (CUDA, CoreML, NNAPI, DirectML) through the provider system.

**Benchmark on target hardware.** Developer machine benchmarks are misleading. A model fast on a workstation can be too slow on a constrained edge device. Always measure p95 and p99 latency, not just mean.

**Know your platform-specific format.** For Android or embedded Linux use TFLite. For Apple devices use CoreML. Both can be exported from ONNX via conversion tools. For cross-platform or server-side edge use ONNX Runtime directly.

**Edge deployment is a design decision, not an afterthought.** Choose a model architecture that fits in the memory and compute budget of the target device before training, not after.